# Integração do dataset e bibliotecas

In [2]:
!pip install anonymizedf

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import hashlib 
import os

path = kagglehub.dataset_download("preritsaxena/fraud-detection")

print("Dataset baixado em:", path)

arquivos = os.listdir(path)
print("Arquivos encontrados:", arquivos)

csv_path = os.path.join(path, arquivos[0]) 

df = pd.read_csv(csv_path)

print("\nFirst 5 records:")
display(df.head())

/home/gabriel/Área de trabalho/fraud_detector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset baixado em: /home/gabriel/.cache/kagglehub/datasets/preritsaxena/fraud-detection/versions/1
Arquivos encontrados: ['Fraud_Data.csv']

First 5 records:


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,is_fraud
0,22058,2015-02-24 22:55:49,2015-04-18 2:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 1:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 7:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


# Verificar dados duplicados

In [3]:
duplicados = df.duplicated(["user_id", "signup_time", "purchase_time", "purchase_value", "device_id", "source", "browser", "sex", "age", "ip_address", "is_fraud"]).sum()
display("A quantidade de dados duplicados é: ", duplicados)

'A quantidade de dados duplicados é: '

np.int64(0)

# Verificar users e devices duplicados

In [4]:
#Esta verificação é importante pois podem haver bots fazendo compras ao mesmo tempo usando o mesmo id de usuario e dispositivo

user_duplicado = df.duplicated(subset=['user_id']).sum()
print(f"Quantidade de usuarios duplicados: {user_duplicado}")

compras_suspeitas = df.duplicated(subset=['purchase_time','device_id']).sum()
print(f"Quantidadede de compras suspeitas: {compras_suspeitas}")


Quantidade de usuarios duplicados: 0
Quantidadede de compras suspeitas: 0


# Formatação de dados.

Verificar dados irrelevantes.

In [5]:
filtro = (
    df['device_id'].str.contains(r'[^a-zA-Z0-9]', na=False) |
    df['source'].str.contains(r'[^a-zA-Z0-9]', na=False) |
    df['browser'].str.contains(r'[^a-zA-Z0-9]', na=False)|
    df['sex'].str.contains(r'[^a-zA-Z0-9]', na=False)
)
print(f"Quantidade de dados irrelevantes: {filtro}")
if filtro.sum() > 0:
    display(df[filtro].head())

Quantidade de dados irrelevantes: 0         False
1         False
2         False
3         False
4         False
          ...  
151107    False
151108    False
151109    False
151110    False
151111    False
Length: 151112, dtype: bool


Remover dados irrelevantes

In [14]:
apagar = (
    df['device_id'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['source'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['browser'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['sex'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['age'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['user_id'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['signup_time'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['purchase_time'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['purchase_value'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['ip_address'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['is_fraud'].replace(r'[^a-zA-Z0-9]','',regex=True )
)

print("Colunas limpas")

Colunas limpas


# Ocultar dados sensiveis

In [7]:
df['sex'] = pd.factorize(df['sex'])[0] # Feminino = 1, Masculino = 0
df['age'] = 'Unknown'
df['ip_address'] = 'Unknown'
df['device_id'] = 'Unknown'
print("Tabela com dados ocultados")
df.head()

Tabela com dados ocultados


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,is_fraud
0,22058,2015-02-24 22:55:49,2015-04-18 2:47:11,34,Unknown,SEO,Chrome,0,Unknown,Unknown,0
1,333320,2015-06-07 20:39:50,2015-06-08 1:38:54,16,Unknown,Ads,Chrome,1,Unknown,Unknown,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,Unknown,SEO,Opera,0,Unknown,Unknown,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,Unknown,SEO,Safari,0,Unknown,Unknown,0
4,221365,2015-07-21 7:09:52,2015-09-09 18:40:53,39,Unknown,Ads,Safari,0,Unknown,Unknown,0
